In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(1)

In [3]:
# Experimental data:
df_exp = pd.DataFrame({
    "Subject": ["S1", "S1", "S2", "S2", "S3", "S3"],
    "Session": [1, 2, 1, 2, 1, 2],
    "RT": np.random.normal(loc=500, scale=50, size=6)
})

# Demographics data:
df_demo = pd.DataFrame({
    "Subject": ["S1", "S2", "S3"],
    "Age": [23, 27, 21],
    "Gender": ["F", "M", "F"]
})

> merge() performs database-style joins using key columns.

Join types:
   - Inner join: Keeps only matching keys in both tables. Equivalent to set intersection.
   - Left join: Keeps all rows from left table; unmatched right values become NaN.
   - Right join: Keeps all rows from right table.
   - Outer join: Keeps all keys from both tables (union); unmatched values filled with NaN.

Basic syntax:
> pd.merge(left_df, right_df, on="key", how="type")

In [4]:
# Let's merge demographics and experimental data.

df_merged = pd.merge(
    df_exp,
    df_demo,
    on="Subject",
    how="left"   # keep all experimental trials
)

print(df_merged)

  Subject  Session          RT  Age Gender
0      S1        1  581.217268   23      F
1      S1        2  469.412179   23      F
2      S2        1  473.591412   27      M
3      S2        2  446.351569   27      M
4      S3        1  543.270381   21      F
5      S3        2  384.923065   21      F


Note: for most experimental pipelines, left is safest when enriching trial-level data.

> concat() stacks DataFrames along an axis.

- Axis 0 → vertical stacking (more rows)
- Axis 1 → horizontal stacking (more columns)

In [5]:
# Let's concatenate two sessions vertically by supposing session 1 and session 2 were stored separately.

df_sess1 = df_exp[df_exp["Session"] == 1]
df_sess2 = df_exp[df_exp["Session"] == 2]

# Vertical concatenation:
df_combined = pd.concat([df_sess1, df_sess2], axis=0) # This stacks rows.

print(df_exp)
print(df_combined)

# Clean indexing:
df_combined = pd.concat([df_sess1, df_sess2], axis=0, ignore_index=True)
print("\ndf_combined index cleaned:\n", df_combined)

  Subject  Session          RT
0      S1        1  581.217268
1      S1        2  469.412179
2      S2        1  473.591412
3      S2        2  446.351569
4      S3        1  543.270381
5      S3        2  384.923065
  Subject  Session          RT
0      S1        1  581.217268
2      S2        1  473.591412
4      S3        1  543.270381
1      S1        2  469.412179
3      S2        2  446.351569
5      S3        2  384.923065

df_combined index cleaned:
   Subject  Session          RT
0      S1        1  581.217268
1      S2        1  473.591412
2      S3        1  543.270381
3      S1        2  469.412179
4      S2        2  446.351569
5      S3        2  384.923065


Note:
- Use concat when:
    - DataFrames share the same structure.
    - You are extending observations, not matching keys.
- Do NOT use merge for vertical stacking.

> join() is similar to merge but index-based by default.

In [7]:
df_demo_indexed = df_demo.set_index("Subject")
df_exp_indexed = df_exp.set_index("Subject")

df_joined = df_exp_indexed.join(df_demo_indexed, how="left")
print(df_joined)

         Session          RT  Age Gender
Subject                                 
S1             1  581.217268   23      F
S1             2  469.412179   23      F
S2             1  473.591412   27      M
S2             2  446.351569   27      M
S3             1  543.270381   21      F
S3             2  384.923065   21      F


##### Handling Duplicate IDs Gracefully

In [38]:
# Duplicated dataframe:

df_demo_dup = pd.DataFrame({
    "Subject": ["S1", "S1", "S2", "S3"], # S1 appears twice in demographics.
    "Age": [25, 26, 31, 29]
})

# Merge
print(pd.merge(df_exp, df_demo_dup, on="Subject", how="left"))

# Detect duplicates:
print(df_demo_dup.duplicated(subset=["Subject"]).any())

# Inspect:
print(df_demo_dup[df_demo_dup.duplicated("Subject", keep=False)])

  Subject  Session          RT  Age
0      S1        1  581.217268   25
1      S1        1  581.217268   26
2      S1        2  469.412179   25
3      S1        2  469.412179   26
4      S2        1  473.591412   31
5      S2        2  446.351569   31
6      S3        1  543.270381   29
7      S3        2  384.923065   29
True
  Subject  Age
0      S1   25
1      S1   26


In [39]:
# Strategies to handle duplicates:

# 1.Remove duplicates:
df_demo_drop = df_demo_dup.drop_duplicates(subset=["Subject"])
print(df_demo_drop)

# 2. Aggregate duplicates before merging:
df_demo_agg = (
    df_demo_dup
    .groupby("Subject", as_index=False)
    .agg({"Age": "mean"})
)
print(df_demo_agg)

# 3. Validate merge expectations:
df_merged= pd.merge(
    df_exp,
    df_demo_clean,
    on="Subject",
    how="left",
    validate="many_to_one"
)
print(df_merged)

  Subject  Age
0      S1   25
2      S2   31
3      S3   29
  Subject   Age
0      S1  25.5
1      S2  31.0
2      S3  29.0
  Subject  Session          RT   Age
0      S1        1  581.217268  25.5
1      S1        2  469.412179  25.5
2      S2        1  473.591412  31.0
3      S2        2  446.351569  31.0
4      S3        1  543.270381  29.0
5      S3        2  384.923065  29.0


validate="many_to_one" enforces that:
- Left side may have many rows per Subject.
- Right side must have one row per Subject.